In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import re
from collections import Counter

In [34]:
### State
class TextState(TypedDict):
    text: str
    count: dict
    lowercase : str
    cleaned_text: str

In [35]:
def lowercase_text(state: TextState)-> TextState:
    text = state["text"]
    lowercase = text.lower()
    return {"lowercase": lowercase}

In [36]:
def text_cleaning(state: TextState)-> TextState:
    lower = state["lowercase"]
    cleaned_text = re.sub(r"[^a-zA-Z0-9\s]", "", lower)
    cleaned_text = " ".join(cleaned_text.split())
    return {"cleaned_text":cleaned_text}

In [37]:
def count_word_frequency(state: TextState):

    cleaned_text = state["cleaned_text"]
    words = cleaned_text.split()

    count = dict(Counter(words))

    return {
        "count": count
    }

In [38]:
workflow = StateGraph(TextState)

In [39]:
# Add nodes
workflow.add_node("text_cleaning", text_cleaning)
workflow.add_node("lowercase_text", lowercase_text)
workflow.add_node("count_word_frequency", count_word_frequency)

In [40]:
# Define sequential flow
workflow.add_edge(START, "lowercase_text")
workflow.add_edge("lowercase_text", "text_cleaning")
workflow.add_edge("text_cleaning", "count_word_frequency")
workflow.add_edge("count_word_frequency", END)


In [41]:
app = workflow.compile()

In [42]:
initial_state = {
    "text": "Hello, World! Hello Python. Python is AMAZING!",
    "cleaned_text": "",
    "lowercase_text": "",
    "count": {}
}

In [43]:
result = app.invoke(initial_state)


In [44]:
print(result)

{'text': 'Hello, World! Hello Python. Python is AMAZING!', 'count': {'hello': 2, 'world': 1, 'python': 2, 'is': 1, 'amazing': 1}, 'lowercase': 'hello, world! hello python. python is amazing!', 'cleaned_text': 'hello world hello python python is amazing'}


In [45]:
print(result["count"])

{'hello': 2, 'world': 1, 'python': 2, 'is': 1, 'amazing': 1}
